# Predicting KRAS Gene Effect: LinearRegression, LightGBM, and XGBoost

The goal is to build a reproducible baseline and compare it with two gradient-boosting models:

```text
model data → feature preparation → model training → KRAS_GeneEffect prediction comparison
```

Each row represents one cell model. The more negative `KRAS_GeneEffect` is, the stronger the KRAS knockout effect.

All preprocessing operations are inside a `Pipeline`. Missing-value imputation, scaling, and encoding are therefore fitted only on the training set and never see the test set.


## 1. Load the data

Use the prepared `KRAS_data_expanded.csv` dataset.


In [ ]:
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Suppress the expected warning: unseen rare categories
# are intentionally mapped by the encoder to the infrequent group.
warnings.filterwarnings(
    "ignore", message="Found unknown categories.*infrequent category",
    category=UserWarning,
)


data = pd.read_csv("/Users/nident/Desktop/JOB/Cancer Cell Line Encyclopedia (CCLE)/data/pilot/KRAS_data_expanded.csv")
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__, "| sklearn:", sklearn.__version__)
print("Table shape:", data.shape)
display(data.head())


## 2. Separate features by type

Numerical features:

- `KRAS_expression` — KRAS expression;
- `Age` — patient age.

Categorical features:

- `KRAS_hotspot`;
- `PrimaryOrMetastasis`;
- `SampleCollectionSite`;
- `Sex`;
- `GrowthPattern`.

Identifiers `ModelID`, `PatientID`, and `CellLineName` are not supplied to the model. `PatientID` is used only for data splitting.

Excluded:

- `KRAS_protein_changes`: there are too many specific variants relative to the 46 rows. They should be tested in a separate experiment after the baseline.


In [ ]:
target = "KRAS_GeneEffect"

numeric_features = [
    "KRAS_expression",
    "Age",
]

categorical_features = [
    "KRAS_hotspot",
    "PrimaryOrMetastasis",
    "SampleCollectionSite",
    "Sex",
    "GrowthPattern",
]

feature_columns = numeric_features + categorical_features
required = ["ModelID", "PatientID", target] + feature_columns
missing_columns = sorted(set(required) - set(data.columns))
if missing_columns:
    raise KeyError(f"KRAS_data_expanded.csv is missing columns: {missing_columns}")

model_data = data[required].copy()
model_data = model_data.dropna(subset=[target]).reset_index(drop=True)

# An unknown PatientID receives its own ModelID group.
groups = model_data["PatientID"].fillna(model_data["ModelID"])

display(pd.DataFrame({
    "dtype": model_data[feature_columns].dtypes.astype(str),
    "missing_n": model_data[feature_columns].isna().sum(),
    "unique_n": model_data[feature_columns].nunique(dropna=True),
}))
print("Models:", len(model_data), "| patient groups:", groups.nunique())


## 3. Split train and test by PatientID

The test set is hidden until final evaluation. All cell lines from one patient remain together, preventing a related line from the same source from appearing in both train and test.

`GroupShuffleSplit` creates one split. On a small dataset, the result depends on the particular test set, so repeated or nested group validation will be needed later.


In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(
    model_data[feature_columns],
    model_data[target],
    groups=groups,
))

train = model_data.iloc[train_idx].copy()
test = model_data.iloc[test_idx].copy()

X_train = train[feature_columns]
y_train = train[target]
groups_train = train["PatientID"].fillna(train["ModelID"])

X_test = test[feature_columns]
y_test = test[target]
groups_test = test["PatientID"].fillna(test["ModelID"])

assert set(groups_train).isdisjoint(set(groups_test))
print("Train:", len(train), "models,", groups_train.nunique(), "patient groups")
print("Test:", len(test), "models,", groups_test.nunique(), "patient groups")
display(pd.DataFrame({
    "train": train["KRAS_hotspot"].value_counts(dropna=False),
    "test": test["KRAS_hotspot"].value_counts(dropna=False),
}).fillna(0).astype(int))


## 4. Create preprocessing transformations

For numerical features:

1. `SimpleImputer(strategy="most_frequent")` fills missing values with the most frequent value from the **training set**;
2. `StandardScaler` puts features on approximately the same scale.

For categorical features:

1. missing values are filled with the most frequent training category;
2. `OneHotEncoder` creates numerical 0/1 columns.

`handle_unknown="infrequent_if_exist"` maps unseen test categories to the infrequent category. `drop="first"` removes one reference category per feature and reduces column collinearity.

Mode imputation is used for this baseline. A dedicated unknown category is often more appropriate for unknown biological statuses and should be compared in a later experiment.


In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        drop="first",
        min_frequency=2,
        sparse_output=False,
    )),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])

pipeline


## 5. Configure GridSearchCV

`LinearRegression` has few meaningful hyperparameters. Test:

- `fit_intercept`: whether an intercept is needed;
- `positive`: whether negative coefficients should be prohibited.

For this task, `positive=True` may be too restrictive because a hotspot can be associated with a lower Gene Effect. GridSearch evaluates this on train. It also tests minimum category frequencies of 2, 3, or 5; rarer categories are grouped together.

Five group-based splits are used inside train. MAE is the optimization metric. Scikit-learn returns negative MAE because GridSearch maximizes its score, so the sign is reversed for display.


In [ ]:
inner_cv = GroupKFold(n_splits=5)

param_grid = {
    "model__fit_intercept": [True, False],
    "model__positive": [False, True],
    "preprocessor__categorical__encoder__min_frequency": [2, 3, 5],
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=inner_cv,
    n_jobs=1,
    return_train_score=True,
)

grid.fit(X_train, y_train, groups=groups_train)

print("Best parameters:", grid.best_params_)
print(f"Mean validation MAE within train: {-grid.best_score_:.3f}")

cv_results = pd.DataFrame(grid.cv_results_)
display(
    cv_results[[
        "params", "mean_train_score", "mean_test_score", "std_test_score", "rank_test_score"
    ]]
    .assign(
        train_MAE=lambda d: -d["mean_train_score"],
        validation_MAE=lambda d: -d["mean_test_score"],
    )
    .sort_values("rank_test_score")
)


## 6. Evaluate on the held-out test set

Compare against `DummyRegressor`, which predicts the mean of `y_train` for every sample.

Metrics:

- MAE — mean absolute error in Gene Effect units;
- RMSE — penalizes large errors more strongly;
- R² — compares the model with mean prediction. It is unstable on a small test set and can be negative.

Lower MAE and RMSE are better. Higher R² is better.


In [ ]:
best_model = grid.best_estimator_
test_prediction = best_model.predict(X_test)

baseline = DummyRegressor(strategy="mean")
baseline.fit(np.zeros((len(y_train), 1)), y_train)
baseline_prediction = baseline.predict(np.zeros((len(y_test), 1)))

def regression_metrics(y_true, prediction):
    return {
        "MAE": mean_absolute_error(y_true, prediction),
        "RMSE": np.sqrt(mean_squared_error(y_true, prediction)),
        "R2": r2_score(y_true, prediction),
    }

metrics = pd.DataFrame({
    "LinearRegression": regression_metrics(y_test, test_prediction),
    "Training mean": regression_metrics(y_test, baseline_prediction),
}).T

display(metrics)
print(
    "MAE change relative to baseline:",
    f"{metrics.loc['Training mean', 'MAE'] - metrics.loc['LinearRegression', 'MAE']:.3f}"
)


## 7. Inspect individual predictions

This helps determine whether the result is driven by a few favorable rows and where the model makes large errors.


In [ ]:
predictions = test[[
    "ModelID", "PatientID", "KRAS_hotspot", "PrimaryOrMetastasis",
    "SampleCollectionSite", target
]].copy()
predictions["Prediction"] = test_prediction
predictions["Baseline_prediction"] = baseline_prediction
predictions["Absolute_error"] = np.abs(predictions[target] - predictions["Prediction"])
predictions = predictions.sort_values("Absolute_error", ascending=False)

display(predictions)

low = min(y_test.min(), test_prediction.min())
high = max(y_test.max(), test_prediction.max())
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, test_prediction, s=55, alpha=0.8)
ax.plot([low, high], [low, high], linestyle="--", color="grey")
ax.set_xlabel("Actual KRAS Gene Effect")
ax.set_ylabel("Predicted KRAS Gene Effect")
ax.set_title("Predictions on held-out models")
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


## 8. Inspect coefficients

Coefficients show the direction of association within this linear model while holding the other features fixed.

- a negative coefficient is associated with a more negative prediction;
- a positive coefficient is associated with a higher prediction;
- a categorical-column coefficient is measured relative to its omitted reference category.

Coefficients estimated from 46 rows are unstable. They do not prove causality and are not a final measure of feature importance.


In [ ]:
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
coefficients = pd.DataFrame({
    "feature": feature_names,
    "coefficient": best_model.named_steps["model"].coef_,
})
coefficients["abs_coefficient"] = coefficients["coefficient"].abs()
display(coefficients.sort_values("abs_coefficient", ascending=False))


## 9. LightGBM

LightGBM builds a sequence of small decision trees rather than a single linear formula. It can therefore detect nonlinear relationships and feature interactions. For example, the influence of `KRAS_expression` may differ between hotspot and non-hotspot models.

For a fair comparison with linear regression, use the same `X_train`, `y_train`, `groups_train`, `X_test`, `y_test`, preprocessing, and grouped `GroupKFold`. Parameters are selected only within train.


In [ ]:
from sklearn.base import clone
from lightgbm import LGBMRegressor

lightgbm_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", LGBMRegressor(
        objective="regression",
        random_state=42,
        n_jobs=1,
        verbosity=-1,
    )),
])

# Small grid: deep trees are unnecessary with so few rows.
lightgbm_param_grid = {
    "model__n_estimators": [50, 100],
    "model__learning_rate": [0.03, 0.10],
    "model__num_leaves": [3, 5, 7],
    "model__min_child_samples": [3, 5, 10],
    "preprocessor__categorical__encoder__min_frequency": [2, 5],
}

lightgbm_grid = GridSearchCV(
    estimator=lightgbm_pipeline,
    param_grid=lightgbm_param_grid,
    scoring="neg_mean_absolute_error",
    cv=inner_cv,
    n_jobs=1,
    return_train_score=True,
)

lightgbm_grid.fit(X_train, y_train, groups=groups_train)
best_lightgbm = lightgbm_grid.best_estimator_
lightgbm_prediction = best_lightgbm.predict(X_test)

print("Best LightGBM parameters:", lightgbm_grid.best_params_)
print(f"Mean validation MAE within train: {-lightgbm_grid.best_score_:.3f}")
display(pd.DataFrame({
    "LightGBM": regression_metrics(y_test, lightgbm_prediction),
    "LinearRegression": regression_metrics(y_test, test_prediction),
    "Training mean": regression_metrics(y_test, baseline_prediction),
}).T)


## 10. XGBoost

XGBoost also builds trees sequentially but uses a different regularization and optimization algorithm. It tests the same hypothesis: whether nonlinear rules predict `KRAS_GeneEffect` better than a linear model and a simple mean.

The grid is intentionally small. With only 37 training samples, a large grid and complex trees can easily overfit random properties of train.


In [ ]:
from xgboost import XGBRegressor

xgboost_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        eval_metric="mae",
        random_state=42,
        n_jobs=1,
        tree_method="hist",
        verbosity=0,
    )),
])

xgboost_param_grid = {
    "model__n_estimators": [50, 100],
    "model__learning_rate": [0.03, 0.10],
    "model__max_depth": [1, 2, 3],
    "model__min_child_weight": [1, 5],
    "model__reg_lambda": [1, 5],
}

xgboost_grid = GridSearchCV(
    estimator=xgboost_pipeline,
    param_grid=xgboost_param_grid,
    scoring="neg_mean_absolute_error",
    cv=inner_cv,
    n_jobs=1,
    return_train_score=True,
)

xgboost_grid.fit(X_train, y_train, groups=groups_train)
best_xgboost = xgboost_grid.best_estimator_
xgboost_prediction = best_xgboost.predict(X_test)

print("Best XGBoost parameters:", xgboost_grid.best_params_)
print(f"Mean validation MAE within train: {-xgboost_grid.best_score_:.3f}")
display(pd.DataFrame({
    "XGBoost": regression_metrics(y_test, xgboost_prediction),
    "LinearRegression": regression_metrics(y_test, test_prediction),
    "Training mean": regression_metrics(y_test, baseline_prediction),
}).T)


## 11. Compare all models on the same test set

Use identical metrics for all models on the same nine held-out samples:

- **MAE** — mean absolute error; lower is better;
- **RMSE** — penalizes large errors more strongly; lower is better;
- **R²** — fraction of explained variance; higher is better. A negative value means the model is worse than the constant training-mean prediction.

The dashed line marks perfect predictions: the closer a point is to the line, the smaller its error.


In [ ]:
prediction_by_model = {
    "LinearRegression": test_prediction,
    "LightGBM": lightgbm_prediction,
    "XGBoost": xgboost_prediction,
    "Training mean": baseline_prediction,
}

comparison_metrics = pd.DataFrame({
    name: regression_metrics(y_test, prediction)
    for name, prediction in prediction_by_model.items()
}).T.sort_values("MAE")
display(comparison_metrics)

comparison_predictions = test[["ModelID", "PatientID", target]].copy()
for name, prediction in prediction_by_model.items():
    comparison_predictions[name] = prediction
display(comparison_predictions)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
tree_predictions = {
    "LinearRegression": test_prediction,
    "LightGBM": lightgbm_prediction,
    "XGBoost": xgboost_prediction,
}
all_values = np.concatenate([
    y_test.to_numpy(), test_prediction, lightgbm_prediction, xgboost_prediction
])
low, high = all_values.min(), all_values.max()

for ax, (name, prediction) in zip(axes, tree_predictions.items()):
    ax.scatter(y_test, prediction, s=55, alpha=0.8)
    ax.plot([low, high], [low, high], linestyle="--", color="grey")
    ax.set_title(f"{name}\nMAE = {comparison_metrics.loc[name, 'MAE']:.3f}")
    ax.set_xlabel("Actual Gene Effect")
    ax.grid(alpha=0.2)
axes[0].set_ylabel("Predicted Gene Effect")
fig.suptitle("Predictions on the same held-out set")
fig.tight_layout()
plt.show()


## 12. Features used by the tree models

`feature_importances_` shows which features the trees used most often to reduce error. This is useful for exploratory analysis but does not prove that a feature causally affects KRAS dependency. With such a small dataset, the importance ranking may change substantially with a different split.


In [ ]:
def feature_importance_table(fitted_pipeline, model_name):
    names = fitted_pipeline.named_steps["preprocessor"].get_feature_names_out()
    values = fitted_pipeline.named_steps["model"].feature_importances_
    return (
        pd.DataFrame({"feature": names, "importance": values})
        .assign(model=model_name)
        .sort_values("importance", ascending=False)
    )

tree_importances = pd.concat([
    feature_importance_table(best_lightgbm, "LightGBM"),
    feature_importance_table(best_xgboost, "XGBoost"),
], ignore_index=True)

display(tree_importances.groupby("model", group_keys=False).head(10))


## 13. Limitations

This pipeline tests technical predictability on a very small dataset. One test set contains about nine models, so conclusions are preliminary.

Key limitations:

- GridSearch selects the best configuration only within train;
- the same test set compares all three models, so it cannot yet identify a final winner;
- one random split can produce unstable results;
- rare collection sites create categories represented by only one or two models;
- metadata may describe model origin without causally affecting KRAS dependency;
- trees can memorize a small training set, so compare train, validation, and test MAE;
- a more reliable estimate requires nested repeated validation grouped by patient;
- Ridge is more suitable for regularizing many features because it has a meaningful `alpha` parameter.
